# FinTok AI · 04_video / 01 · Montador de vídeo (demo)

**Objetivo:** probar el montador (`src/video/compositor.py`) con un guion de prueba antes de tener el pipeline de texto y voz real.

**Entrada:**
- `compositor.py`
- Los avatares: `andrea_closed.png`, `andrea_half.png`, `andrea_open.png`
- Un JSON de escenas (aquí lo creamos de prueba, con voces generadas con gTTS)

**Salida:** un vídeo vertical 1080×1920 en `outputs/demo.mp4`.

No necesita GPU.

## 0. Instalación

In [1]:
!pip install -q gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 2.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.31.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


## 1. Subir el montador y los avatares

Selecciona a la vez `compositor.py` y los tres PNG del avatar.

In [3]:
import os, shutil, json
from google.colab import files

uploaded = files.upload()
os.makedirs("assets/avatar", exist_ok=True)
for name in uploaded:
    if name.endswith(".png"):
        shutil.move(name, f"assets/avatar/{name}")
print(sorted(os.listdir("assets/avatar")))

Saving andrea_half.png to andrea_half.png
Saving andrea_preview.gif to andrea_preview.gif
Saving andrea_closed.png to andrea_closed.png
Saving andrea_meta.json to andrea_meta.json
Saving andrea_open.png to andrea_open.png
['andrea_closed.png', 'andrea_half.png', 'andrea_open.png']


## 2. Guion de prueba con voces generadas

Simulamos lo que entregará la parte de texto y audio: una frase por escena, con su audio WAV.
Las cifras son de relleno, solo para probar.

In [4]:
from gtts import gTTS

scenes = [
    {"scene_id": 1, "scene_type": "hook", "speaker": "andrea",
     "text": "NVIDIA ha más que duplicado sus ingresos en un solo año.",
     "key_figure": "x2", "source": "NVIDIA 10-K FY2025", "verified": True},
    {"scene_id": 2, "scene_type": "dato", "speaker": "andrea",
     "text": "El negocio de data center ya supone la gran mayoría de las ventas.",
     "key_figure": "~88%", "source": "NVIDIA 10-K FY2025", "verified": True},
    {"scene_id": 3, "scene_type": "cierre", "speaker": "andrea",
     "text": "¿El riesgo? Una fuerte concentración en pocos clientes.",
     "key_figure": None, "source": "NVIDIA 10-K FY2025", "verified": True},
]

os.makedirs("mock", exist_ok=True)
for s in scenes:
    mp3 = f"mock/scene_{s['scene_id']:02d}.mp3"
    wav = mp3.replace(".mp3", ".wav")
    gTTS(s["text"], lang="es").save(mp3)
    os.system(f"ffmpeg -y -loglevel error -i {mp3} -ar 24000 -ac 1 {wav}")
    s["audio_path"] = wav

spec = {"company": "NVIDIA", "year": 2025, "language": "es", "scenes": scenes}
with open("mock/scenes.json", "w", encoding="utf-8") as f:
    json.dump(spec, f, ensure_ascii=False, indent=2)
print(json.dumps(spec, ensure_ascii=False, indent=2)[:800])

{
  "company": "NVIDIA",
  "year": 2025,
  "language": "es",
  "scenes": [
    {
      "scene_id": 1,
      "scene_type": "hook",
      "speaker": "andrea",
      "text": "NVIDIA ha más que duplicado sus ingresos en un solo año.",
      "key_figure": "x2",
      "source": "NVIDIA 10-K FY2025",
      "verified": true,
      "audio_path": "mock/scene_01.wav"
    },
    {
      "scene_id": 2,
      "scene_type": "dato",
      "speaker": "andrea",
      "text": "El negocio de data center ya supone la gran mayoría de las ventas.",
      "key_figure": "~88%",
      "source": "NVIDIA 10-K FY2025",
      "verified": true,
      "audio_path": "mock/scene_02.wav"
    },
    {
      "scene_id": 3,
      "scene_type": "cierre",
      "speaker": "andrea",
      "text": "¿El riesgo? Una fuerte concentra


## 3. Montar el vídeo

Tarda unos segundos por cada segundo de vídeo.
Para probar fondos, añade a cualquier escena `"background_path": "ruta/imagen.png"`; si no hay, usa un degradado.

In [6]:
try:
    from src.video.compositor import render_video, Style   # si se ejecuta dentro del repo
except ModuleNotFoundError:
    from compositor import render_video, Style             # si se ha subido el .py suelto (Colab)

out = render_video("mock/scenes.json", avatar_dir="assets/avatar", out_path="outputs/demo.mp4")
print("Vídeo:", out)

Vídeo: outputs/demo.mp4


In [7]:
from IPython.display import Video
Video("outputs/demo.mp4", embed=True, width=360)

## 4. Ajustes de estilo

Todo lo visual está en la clase `Style`: colores, tamaño y posición de la burbuja, umbrales de volumen para abrir la boca, aviso legal…

In [ ]:
from src.video.compositor import Style

estilo = Style(accent=(255, 196, 0), bubble_d=620, mouth_thresholds=(0.10, 0.35))
render_video("mock/scenes.json", avatar_dir="assets/avatar", out_path="outputs/demo_estilo.mp4", style=estilo)
Video("outputs/demo_estilo.mp4", embed=True, width=360)

In [ ]:
files.download("outputs/demo.mp4")